# demetrapy: Batch Review and HTML Report

Run a detailed TRAMO/SEATS batch, identify series that need review, inspect one fitted model, and export a self-contained interactive report.

Requires Python 3.11+, Java 9+, and a successful `demetrapy check`.

## 1. Copy: load a small batch

Use three built-in monthly emissions series so the complete workflow stays quick to rerun.

In [1]:
from pathlib import Path

import pandas as pd

from demetrapy import TramoSeatsConfig, adjust_dataframe, load_monthly_emissions


data = load_monthly_emissions()[["power", "transport", "industry"]]
data.tail(3)

,power,transport,industry
date,,,
2024-10-01,70.690678,69.750071,42.023763
2024-11-01,76.088743,67.167389,44.260454
2024-12-01,81.679499,62.826435,47.467286


## 2. Run: adjust every series

`detailed=True` retains fitted models, diagnostics, processing messages, SI output, and the information required by the HTML report. `prediction_length=12` adds one year of monthly forecasts.

In [2]:
config = TramoSeatsConfig(
    spec="RSAfull",
    preprocessing={"automodel": {"enabled": True}},
    seats={"prediction_length": 12},
)

result = adjust_dataframe(data, config=config, detailed=True)
print(f"Adjusted {len(result.results)} series with TRAMO/SEATS")

Adjusted 3 series with TRAMO/SEATS


## 3. Inspect: build a review queue

The summary has one row per input series. Counts are navigation aids: diagnostics provide model evidence, while processing messages are not necessarily warnings.

In [3]:
summary = result.to_summary_frame()
summary

,series,method,specification,arima,automatic,diagnostic_count,message_count,forecast_periods
0,power,tramoseats,RSAfull,"ARIMA(0,0,0)(0,1,1)[12]",True,126,1,12
1,transport,tramoseats,RSAfull,"ARIMA(0,1,1)(1,1,0)[12]",True,126,1,12
2,industry,tramoseats,RSAfull,"ARIMA(0,1,1)(0,1,1)[12]",True,124,1,12


In [4]:
follow_up = summary.loc[
    (summary["diagnostic_count"] > 0) | (summary["message_count"] > 0),
    ["series", "arima", "automatic", "diagnostic_count", "message_count"],
]
follow_up

,series,arima,automatic,diagnostic_count,message_count
0,power,"ARIMA(0,0,0)(0,1,1)[12]",True,126,1
1,transport,"ARIMA(0,1,1)(1,1,0)[12]",True,126,1
2,industry,"ARIMA(0,1,1)(0,1,1)[12]",True,124,1


## 4. Inspect one fitted model

Change `target` and rerun the next cell to inspect another series. The diagnostic mapping comes directly from JDemetra+.

In [5]:
target = "power"  # Try "transport" or "industry".
series_result = result.for_series(target)
model = series_result.arima_model

print("Method:", series_result.method)
print("Specification:", series_result.specification)
print("ARIMA:", model.notation if model is not None else "unavailable")

pd.DataFrame(
    list(series_result.diagnostics.items())[:10],
    columns=["diagnostic", "value"],
)

Method: tramoseats
Specification: RSAfull
ARIMA: ARIMA(0,0,0)(0,1,1)[12]


,diagnostic,value
0,preprocessing.period,12
1,preprocessing.span.n,120
2,preprocessing.span.missing,0
3,preprocessing.espan.n,120
4,preprocessing.log,0
5,preprocessing.regression.ntd,0
6,preprocessing.regression.nmh,0
7,preprocessing.regression.nout,0
8,preprocessing.regression.noutao,0
9,preprocessing.regression.noutls,0


In [6]:
messages = pd.DataFrame(
    [
        {
            "type": message.type,
            "name": message.name,
            "origin": message.origin,
            "message": message.message,
        }
        for message in series_result.messages
    ]
)
messages if not messages.empty else "No processing messages"

,type,name,origin,message
0,Warning,decomposition.Model decomposition,ec.satoolkit.seats.DefaultModelDecomposer,Parameters cut off


## 5. Export: create the analyst report

The report combines the batch summary with interactive component, forecast, and SI charts plus expandable diagnostics and processing messages.

In [7]:
project_root = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
)

report_path = result.to_html_report(
    project_root / "example_output" / "tramoseats_batch_review.html",
    title="Monthly Emissions TRAMO/SEATS Review",
)

assert report_path.is_file()
print(f"Open the self-contained report: {report_path.resolve()}")

Open the self-contained report: /Users/sermetpekin/Desktop/git_repos/seasonal-pri/example_output/tramoseats_batch_review.html


## Where to go next

The HTML report embeds Plotly, so it can be shared and opened without a server or internet connection. Its larger file size is the tradeoff for portability.

- [Usage guide](../../docs/USAGE.md): detailed results, summary frames, and report options.
- [Batch summary script](../15_batch_summary.py): compact automation and CSV export.
- [HTML report script](../16_html_report.py): non-notebook version of the report workflow.